In [1]:

import pandas as pd
import numpy as np

df = pd.read_csv("amazon.csv")

print("Dataset shape:", df.shape)
print("\nDuplicate product IDs:", df['product_id'].duplicated().sum())
print("\nMissing values in selected columns:")
print(df[
    ['product_id', 'product_name', 'rating',
     'rating_count', 'discounted_price', 'discount_percentage']
].isnull().sum())

print("\nSample records:")
print(df[
    ['product_id', 'product_name', 'rating',
     'rating_count', 'discounted_price', 'discount_percentage']
].head())


Dataset shape: (1465, 16)

Duplicate product IDs: 114

Missing values in selected columns:
product_id             0
product_name           0
rating                 0
rating_count           2
discounted_price       0
discount_percentage    0
dtype: int64

Sample records:
   product_id                                       product_name rating  \
0  B07JW9H4J1  Wayona Nylon Braided USB to Lightning Fast Cha...    4.2   
1  B098NS6PVG  Ambrane Unbreakable 60W / 3A Fast Charging 1.5...    4.0   
2  B096MSW6CT  Sounce Fast Phone Charging Cable & Data Sync U...    3.9   
3  B08HDJ86NZ  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...    4.2   
4  B08CF3B7N1  Portronics Konnect L 1.2M Fast Charging 3A 8 P...    4.2   

  rating_count discounted_price discount_percentage  
0       24,269             ₹399                 64%  
1       43,994             ₹199                 43%  
2        7,928             ₹199                 90%  
3       94,363             ₹329                 53%  
4       

In [2]:
ranking_df = df[
    ['product_id', 'product_name', 'rating',
     'rating_count', 'discounted_price', 'discount_percentage']
].copy()
ranking_df['rating'] = pd.to_numeric(
    ranking_df['rating'], errors='coerce'
)
ranking_df['rating_count'] = pd.to_numeric(
    ranking_df['rating_count']
    .astype(str)
    .str.replace(',', '', regex=False),
    errors='coerce'
)
ranking_df['discounted_price'] = pd.to_numeric(
    ranking_df['discounted_price']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False),
    errors='coerce'
)
ranking_df['discount_percentage'] = pd.to_numeric(
    ranking_df['discount_percentage']
    .astype(str)
    .str.replace('%', '', regex=False),
    errors='coerce'
)

print("Missing values after conversion:")
print(ranking_df.isnull().sum())
ranking_df = ranking_df.dropna(
    subset=[
        'product_id', 'rating', 'rating_count',
        'discounted_price', 'discount_percentage'
    ]
).copy()
ranking_df = ranking_df.drop_duplicates(
    subset='product_id',
    keep='first'
).copy()

print("\nCleaned dataset shape:", ranking_df.shape)
print("Remaining duplicate IDs:",
      ranking_df['product_id'].duplicated().sum())

print("\nFeature summary:")
print(
    ranking_df[
        ['rating', 'rating_count',
         'discounted_price', 'discount_percentage']
    ].describe().round(2)
)


Missing values after conversion:
product_id             0
product_name           0
rating                 1
rating_count           2
discounted_price       0
discount_percentage    0
dtype: int64

Cleaned dataset shape: (1348, 6)
Remaining duplicate IDs: 0

Feature summary:
        rating  rating_count  discounted_price  discount_percentage
count  1348.00       1348.00           1348.00              1348.00
mean      4.09      17656.85           3310.27                46.66
std       0.30      42158.84           7180.90                21.61
min       2.00          2.00             39.00                 0.00
25%       3.90       1107.50            349.00                31.00
50%       4.10       4740.00            899.00                49.00
75%       4.30      16051.50           2184.00                62.00
max       5.00     426973.00          77990.00                94.00


In [3]:

C = ranking_df['rating'].mean()
m = ranking_df['rating_count'].quantile(0.25)
ranking_df['adjusted_rating'] = (
    (ranking_df['rating_count'] /
     (ranking_df['rating_count'] + m))
    * ranking_df['rating']
    +
    (m /
     (ranking_df['rating_count'] + m))
    * C
)

print("Overall average rating (C):", round(C, 3))
print("Minimum rating-count threshold (m):", round(m, 2))

print("\nExample products:")
print(
    ranking_df[
        ['product_name', 'rating', 'rating_count', 'adjusted_rating']
    ].sort_values('rating_count').head(10).round(3).to_string(index=False)
)


Overall average rating (C): 4.092
Minimum rating-count threshold (m): 1107.5

Example products:
                                                                                                                                                                                          product_name  rating  rating_count  adjusted_rating
                                                                                                                                                 Khaitan ORFin Fan heater for Home and kitchen-K0 2215     2.0           2.0            4.088
                           NGI Store 2 Pieces Pet Hair Removers for Your Laundry Catcher Lint Remover for Washing Machine Lint Remover Reusable Portable Silica Gel Clothes Washer Dryer Floating Ball     3.1           2.0            4.090
                                                                                                                                       Longway Blaze 2 Rod Quartz Room Heater (White, Gray, 80

In [4]:

from sklearn.preprocessing import MinMaxScaler

ranking_df['log_rating_count'] = np.log1p(ranking_df['rating_count'])
ranking_df['log_price'] = np.log1p(ranking_df['discounted_price'])
scaler = MinMaxScaler()
ranking_df[
    ['rating_score', 'popularity_score',
     'discount_score', 'price_score']
] = scaler.fit_transform(
    ranking_df[
        ['adjusted_rating', 'log_rating_count',
         'discount_percentage', 'log_price']
    ]
)
ranking_df['price_score'] = 1 - ranking_df['price_score']
weights = {
    'rating_score': 0.50,
    'popularity_score': 0.20,
    'discount_score': 0.20,
    'price_score': 0.10
}
ranking_df['ranking_score'] = sum(
    ranking_df[feature] * weight
    for feature, weight in weights.items()
)
print("Ranking score summary:")
print(ranking_df['ranking_score'].describe().round(3))


Ranking score summary:
count    1348.000
mean        0.545
std         0.084
min         0.244
25%         0.487
50%         0.538
75%         0.601
max         0.806
Name: ranking_score, dtype: float64


In [5]:

top10_initial = ranking_df.sort_values(
    'ranking_score',
    ascending=False
).head(10).copy()

top10_initial.insert(0, 'rank', range(1, len(top10_initial) + 1))

print("TOP 10 PRODUCTS — INITIAL WEIGHTS")

print(
    top10_initial[
        ['rank', 'product_id', 'product_name',
         'rating', 'rating_count', 'discounted_price',
         'discount_percentage', 'adjusted_rating',
         'ranking_score']
    ].round(3).to_string(index=False)
)


TOP 10 PRODUCTS — INITIAL WEIGHTS
 rank product_id                                                                                                                                                                         product_name  rating  rating_count  discounted_price  discount_percentage  adjusted_rating  ranking_score
    1 B014I8SX4Y                                                                                                                           Amazon Basics High-Speed HDMI Cable, 6 Feet (2-Pack),Black     4.4      426973.0             309.0                   78            4.399          0.806
    2 B00NH11PEY                                                                  AmazonBasics USB 2.0 - A-Male to A-Female Extension Cable for Personal Computer, Printer (Black, 9.8 Feet/3 Meters)     4.5       74976.0             199.0                   73            4.494          0.804
    3 B00NH11KIK                                                                           Am

In [6]:

alternative_weights = {
    'rating_score': 0.45,
    'popularity_score': 0.25,
    'discount_score': 0.20,
    'price_score': 0.10
}
ranking_df['alternative_score'] = sum(
    ranking_df[feature] * weight
    for feature, weight in alternative_weights.items()
)
top10_alternative = ranking_df.sort_values(
    'alternative_score',
    ascending=False
).head(10).copy()
initial_ids = set(top10_initial['product_id'])
alternative_ids = set(top10_alternative['product_id'])

common_ids = initial_ids & alternative_ids

print("Products in both top-10 lists:", len(common_ids), "out of 10")
print("Top-10 overlap percentage:", len(common_ids) * 10, "%")

print("\nProducts in the initial top 10:")
print(top10_initial[['product_id', 'product_name']].to_string(index=False))

print("\nProducts in the alternative top 10:")
print(
    top10_alternative[
        ['product_id', 'product_name', 'alternative_score']
    ].to_string(index=False)
)


Products in both top-10 lists: 10 out of 10
Top-10 overlap percentage: 100 %

Products in the initial top 10:
product_id                                                                                                                                                                         product_name
B014I8SX4Y                                                                                                                           Amazon Basics High-Speed HDMI Cable, 6 Feet (2-Pack),Black
B00NH11PEY                                                                  AmazonBasics USB 2.0 - A-Male to A-Female Extension Cable for Personal Computer, Printer (Black, 9.8 Feet/3 Meters)
B00NH11KIK                                                                           AmazonBasics USB 2.0 Cable - A-Male to B-Male - for Personal Computer, Printer- 6 Feet (1.8 Meters), Black
B07Q4QV1DL                                                              ELV Aluminum Adjustable Mobile Phone Foldable Tabl